# Listening Test — Dataset Samples (14/28)

This notebook prepares the **dataset half** of a listening test. Fourteen clips come from the ANIMA 53-EDO corpus (this notebook); fourteen more will come from Model A and Model B generations (separate notebook, after training).

Each clip is a **16-bar chord-progression excerpt** rendered from an MPE MIDI file using the sine-wave renderer (`src/play_mpe.py`). Sine waves are chosen intentionally — they preserve the 53-EDO microtuning cleanly (no detuned piano samples masking commas).

**Why 14?** The dataset provides **14 distinct voicing/tuning transformations** of every song: `type_0_major`, `type_0_minor`, `type_1_minor`, `type_1_neutral`, `type_2_minor`, `type_2_subminor`, `type_3_major`, `type_3_minor`, `type_4_minor`, `type_4_upmajor`, `type_5_major_v2`, `type_5_minor`, `type_6_minor`, `type_6_neutral_n`. `type_0_major` is the 12-TET reference; the other 13 use genuine 53-EDO microtonal reharmonisations. Testing only `type_0_major` would defeat the purpose of a microtonal dataset, so each of the 14 songs is assigned a **distinct transformation** — every clip probes a different tuning.

**Survey questions** (per clip):
- **Harmony** — How coherent do you find the harmonic motion?
- **Plausibility** — How plausible is this chord progression for a potential song?
- **Dissonance** — How dissonant is this chord progression?
- **Novelty** — How surprising or novel do you find this progression?

**Output**: `dataset/audio/listening_test/*.wav`, `dataset/listening_test/midi/*.mid`, `dataset/listening_test/manifest.json`.

In [85]:
import json
import sys
from pathlib import Path

import numpy as np
from IPython.display import Audio, display

# Make the project source importable regardless of launch directory.
_SRC = Path("/home/david/Projects/ANIMA_Microtonal_GPT/src")
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from tokenizer import parse_mpe_midi, MPETokenizer
from play_mpe import render_mpe_to_audio_data

ROOT = _SRC.parent
MIDI_ROOT_BASE = ROOT / "dataset" / "midi_files" / "53_tet_mpe"

# Audio goes to dataset/audio/listening_test/; trimmed MIDI + manifest stay under dataset/listening_test/.
OUT_AUDIO = ROOT / "dataset" / "audio" / "listening_test"
OUT_DIR   = ROOT / "dataset" / "listening_test"
OUT_MIDI  = OUT_DIR / "midi"
OUT_AUDIO.mkdir(parents=True, exist_ok=True)
OUT_MIDI.mkdir(parents=True, exist_ok=True)

print("MIDI_ROOT_BASE :", MIDI_ROOT_BASE)
print("OUT_AUDIO      :", OUT_AUDIO)
print("OUT_MIDI       :", OUT_MIDI)
assert MIDI_ROOT_BASE.exists(), f"Not found: {MIDI_ROOT_BASE}"

MIDI_ROOT_BASE : /home/david/Projects/ANIMA_Microtonal_GPT/dataset/midi_files/53_tet_mpe
OUT_AUDIO      : /home/david/Projects/ANIMA_Microtonal_GPT/dataset/audio/listening_test
OUT_MIDI       : /home/david/Projects/ANIMA_Microtonal_GPT/dataset/listening_test/midi


## Curated 14-song selection — one transformation each

Fourteen songs chosen to span genres and tonalities, each paired with a **different** 53-EDO voicing/tuning transformation so the listening test samples the whole microtonal vocabulary of the corpus rather than only the 12-TET reference.

| # | Song | Key | Genre | Transformation |
|---|------|-----|-------|----------------|
| 1 | Autumn Leaves            | Cm | Jazz swing (ii-V-i)     | `type_0_major` (12-TET reference) |
| 2 | Stella By Starlight      | C  | Jazz ballad             | `type_0_minor` |
| 3 | Wave                     | C  | Bossa nova              | `type_1_minor` |
| 4 | Something (Beatles)      | C  | Pop/rock ballad         | `type_1_neutral` |
| 5 | Fix You (Coldplay)       | C  | Pop/rock ballad         | `type_2_minor` |
| 6 | Beatrice (Rivers)        | C  | Post-bop jazz           | `type_2_subminor` |
| 7 | So Tinha De Ser Com Voce | C  | Bossa nova / MPB        | `type_3_major` |
| 8 | Lullaby Of Birdland      | C  | Jazz vocal standard     | `type_3_minor` |
| 9 | Thriller (Jackson)       | Cm | Pop                     | `type_4_minor` |
| 10| James (Metheny)          | C  | Contemporary jazz       | `type_4_upmajor` |
| 11| Michelle (Beatles)       | C  | Pop/rock ballad         | `type_5_major_v2` |
| 12| Misty                    | C  | Jazz ballad             | `type_5_minor` |
| 13| Giant Steps              | C  | Bop (Coltrane changes)  | `type_6_minor` |
| 14| Every Time We Say Goodbye (Cole Porter) | C | Jazz standard | `type_6_neutral_n` |

In [86]:
# Each entry pairs one song with one of the 14 transformation types.
SELECTION = [
    # (display_name, filename_glob_fragment, style_tag, transformation_type)
    ("Autumn Leaves",         "_Autumn Leaves_F_",    "jazz_swing",       "type_0_major"),
    ("Stella By Starlight",   "_Stella By Starlight_F_", "jazz_ballad",   "type_0_minor"),
    ("Wave",                  "_Wave_F_",             "bossa",            "type_1_minor"),
    ("Something",             "_Something_F_",        "pop_rock",         "type_1_neutral"),
    ("Fix You",               "_Fix You_F_",          "pop_rock",         "type_2_minor"),
    ("Beatrice",              "_Beatrice_F_",         "post_bop",         "type_2_subminor"),
    ("So Tinha De Ser Com Voce", "_So Tinha De Ser Com Vo\u00e7e_F_", "bossa_mpb", "type_3_major"),
    ("Lullaby Of Birdland",   "_Lullaby Of Birdland_F_", "jazz_vocal",   "type_3_minor"),
    ("Thriller",              "_Thriller_F_",         "pop",              "type_4_minor"),
    ("James",                 "_James_F_",            "contemporary_jazz","type_4_upmajor"),
    ("Michelle",              "_Michelle_F_",         "pop_rock",         "type_5_major_v2"),
    ("Misty",                 "_Misty_F_",            "jazz_ballad",      "type_5_minor"),
    ("Giant Steps",           "_Giant Steps_F_",      "jazz_bop",         "type_6_minor"),
    ("Every Time We Say Goodbye", "_Every Time We Say Goodbye_F_", "jazz_standard", "type_6_neutral_n"),
]

def resolve(glob_fragment: str, transformation: str) -> Path:
    """Find the MIDI whose stem contains `glob_fragment` inside the given transformation folder."""
    midi_dir = MIDI_ROOT_BASE / transformation
    hits = sorted(
        p for p in midi_dir.iterdir()
        if p.suffix == ".mid" and glob_fragment in p.name
    )
    if not hits:
        raise FileNotFoundError(f"No MIDI matching {glob_fragment!r} in {midi_dir}")
    # Prefer the shortest matching name (no " 1", " 2" variants).
    hits.sort(key=lambda p: (len(p.name), p.name))
    return hits[0]

resolved = []
for name, frag, style, ttype in SELECTION:
    path = resolve(frag, ttype)
    resolved.append((name, path, style, ttype))
    print(f"{name:24s} [{ttype:17s}] -> {path.name}")

Autumn Leaves            [type_0_major     ] -> 37493_Autumn Leaves_F_minor_type_0_major.mid
Stella By Starlight      [type_0_minor     ] -> 05237_Stella By Starlight_F_major_type_0_minor.mid
Wave                     [type_1_minor     ] -> 33137_Wave_F_major_type_1_minor.mid
Something                [type_1_neutral   ] -> 27809_Something_F_major_type_1_neutral.mid
Fix You                  [type_2_minor     ] -> 04097_Fix You_F_major_type_2_minor.mid
Beatrice                 [type_2_subminor  ] -> 06449_Beatrice_F_major_type_2_subminor.mid
So Tinha De Ser Com Voce [type_3_major     ] -> 23825_So Tinha De Ser Com Voçe_F_major_type_3_major.mid
Lullaby Of Birdland      [type_3_minor     ] -> 08321_Lullaby Of Birdland_F_major_type_3_minor.mid
Thriller                 [type_4_minor     ] -> 47765_Thriller_F_minor_type_4_minor.mid
James                    [type_4_upmajor   ] -> 43961_James_F_major_type_4_upmajor.mid
Michelle                 [type_5_major_v2  ] -> 05117_Michelle_F_major_type_5

## Inspect chord structure

Each MIDI is parsed into a list of chord events — `{onset_beats, duration_beats, notes:[{step_53, velocity}]}` — using the read-only `parse_mpe_midi` from `tokenizer.py`. This lets us trim cleanly on chord boundaries.

In [87]:
# Peek at one song so we understand the chord-event format.
name, path, _, ttype = resolved[0]
chords = parse_mpe_midi(path)
total_beats = chords[-1]["onset_beats"] + chords[-1]["duration_beats"]
print(f"{name} [{ttype}]: {len(chords)} chords, total={total_beats:.1f} beats")
for c in chords[:4]:
    steps = [n["step_53"] for n in c["notes"]]
    print(f"  onset={c['onset_beats']:6.2f}  dur={c['duration_beats']:4.1f}  steps={steps}")

Autumn Leaves [type_0_major]: 78 chords, total=576.0 beats
  onset=  0.00  dur= 8.0  steps=[203, 234, 256, 270, 301]
  onset=  8.00  dur= 8.0  steps=[172, 225, 256, 270, 296]
  onset= 16.00  dur= 8.0  steps=[195, 225, 248, 265, 296]
  onset= 24.00  dur= 8.0  steps=[164, 217, 248, 265, 287]


## Trim to 16 bars & render at 160 BPM

iReal-derived MIDI uses 4/4 throughout and 1 beat = 1 quarter note → 16 bars = **64 beats**. We trim every chord whose onset is strictly before `max_beats` and clip the final chord's duration so it never rings past the window.

We write the trimmed MIDI at **160 BPM** (instead of the 120 BPM the clips were packed at) so the progressions have a motion-forward feel rather than dragging.

In [ ]:
N_BARS = 17
BEATS_PER_BAR = 4
MAX_BEATS = N_BARS * BEATS_PER_BAR

TEMPO_BPM = 170                 # render at 160 BPM — jazz/pop standards sound alive here
RENDER_SPEED = 1.0              # no further tempo scaling at render time
RENDER_WAVEFORM = "clarinet"      # clarinet-like timbre for a more natural sound
RENDER_REVERB = 33              # light hall ambience (0 = dry)
SAMPLE_RATE = 44100

    
def trim_chords_to_bars(chords, max_beats=MAX_BEATS):
    """Keep chords starting before `max_beats`; clip the final chord's ring-out."""
    kept = []
    for c in chords:
        if c["onset_beats"] >= max_beats:
            break
        end = c["onset_beats"] + c["duration_beats"]
        if end > max_beats:
            c = dict(c, duration_beats=round(max_beats - c["onset_beats"], 4))
        kept.append(c)
    return kept


def write_trimmed_midi(chords, out_path: Path, tempo_bpm=TEMPO_BPM):
    """Write the chord list back to MPE MIDI at the requested tempo."""
    tok = MPETokenizer()
    tok.chords_to_midi(chords, out_path, tpb=960, tempo_bpm=tempo_bpm)


def render_wav(midi_path: Path, wav_path: Path):
    """Render an MPE MIDI file to a WAV at SAMPLE_RATE using sine synthesis.

    Delegates to play_mpe.render_mpe_to_audio_data's own `save_path` so the
    (channels, samples) → (samples, channels) transpose is handled correctly.
    """
    audio, sr = render_mpe_to_audio_data(
        str(midi_path),
        sample_rate=SAMPLE_RATE,
        speed=RENDER_SPEED,
        waveform=RENDER_WAVEFORM,
        reverb=RENDER_REVERB,
        save_path=str(wav_path),
    )
    if audio is None:
        raise RuntimeError(f"render_mpe_to_audio_data returned None for {midi_path}")
    n_samples = audio.shape[-1] if audio.ndim == 2 else len(audio)
    return sr, n_samples

In [89]:
manifest_entries = []

for idx, (display_name, src_midi, style, ttype) in enumerate(resolved, start=1):
    # 1) parse → trim to 16 bars
    chords = parse_mpe_midi(src_midi)
    trimmed = trim_chords_to_bars(chords)
    actual_end = trimmed[-1]["onset_beats"] + trimmed[-1]["duration_beats"] if trimmed else 0.0

    # 2) write trimmed MIDI at 160 BPM
    safe_name = display_name.replace(" ", "_").replace("'", "")
    stem = f"dataset_{idx:02d}_{safe_name}__{ttype}"
    midi_out = OUT_MIDI / f"{stem}.mid"
    write_trimmed_midi(trimmed, midi_out)

    # 3) render WAV
    wav_out = OUT_AUDIO / f"{stem}.wav"
    sr, n_samples = render_wav(midi_out, wav_out)
    duration_sec = n_samples / sr

    manifest_entries.append({
        "id": f"dataset_{idx:02d}",
        "source": "dataset",
        "song": display_name,
        "style": style,
        "transformation": ttype,
        "source_midi": str(src_midi.relative_to(ROOT)),
        "trimmed_midi": str(midi_out.relative_to(ROOT)),
        "audio": str(wav_out.relative_to(ROOT)),
        "bars": N_BARS,
        "beats_per_bar": BEATS_PER_BAR,
        "tempo_bpm": TEMPO_BPM,
        "n_chords": len(trimmed),
        "end_beat": round(actual_end, 2),
        "duration_sec": round(duration_sec, 2),
        "render": {
            "waveform": RENDER_WAVEFORM,
            "speed": RENDER_SPEED,
            "reverb_pct": RENDER_REVERB,
            "sample_rate": sr,
        },
    })

print(f"\nRendered {len(manifest_entries)} clips.")

play_mpe | waveform: clarinet, reverb: 40%
Loading dataset_01_Autumn_Leaves__type_0_major.mid...
Rendering 46 notes. Total duration: 24.90s (Speed: 1.0x)
Saved: dataset_01_Autumn_Leaves__type_0_major.wav
play_mpe | waveform: clarinet, reverb: 40%
Loading dataset_02_Stella_By_Starlight__type_0_minor.mid...
Rendering 46 notes. Total duration: 24.90s (Speed: 1.0x)
Saved: dataset_02_Stella_By_Starlight__type_0_minor.wav
play_mpe | waveform: clarinet, reverb: 40%
Loading dataset_03_Wave__type_1_minor.mid...
Rendering 61 notes. Total duration: 24.90s (Speed: 1.0x)
Saved: dataset_03_Wave__type_1_minor.wav
play_mpe | waveform: clarinet, reverb: 40%
Loading dataset_04_Something__type_1_neutral.mid...
Rendering 74 notes. Total duration: 24.90s (Speed: 1.0x)
Saved: dataset_04_Something__type_1_neutral.wav
play_mpe | waveform: clarinet, reverb: 40%
Loading dataset_05_Fix_You__type_2_minor.mid...
Rendering 103 notes. Total duration: 24.90s (Speed: 1.0x)
Saved: dataset_05_Fix_You__type_2_minor.wav
p

## Manifest + inline preview

The manifest is written as `dataset/listening_test/manifest.json`. When we add the 14 model clips later, we'll append to the same file so the survey tool reads one list.

Each survey item carries four rating placeholders (Harmony, Plausibility, Dissonance, Novelty) that the downstream test form will populate.

In [90]:
SURVEY_QUESTIONS = [
    {"key": "harmony",      "label": "Harmony",      "prompt": "How coherent do you find the harmonic motion?"},
    {"key": "plausibility", "label": "Plausibility", "prompt": "How plausible is this chord progression for a potential song?"},
    {"key": "dissonance",   "label": "Dissonance",   "prompt": "How dissonant is this chord progression?"},
    {"key": "novelty",      "label": "Novelty",      "prompt": "How surprising or novel do you find this progression?"},
]

for e in manifest_entries:
    e["ratings"] = {q["key"]: None for q in SURVEY_QUESTIONS}

manifest = {
    "tuning": "53-EDO (MPE)",
    "n_bars": N_BARS,
    "tempo_bpm": TEMPO_BPM,
    "survey_questions": SURVEY_QUESTIONS,
    "clips": manifest_entries,
}

manifest_path = OUT_DIR / "manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(f"wrote {manifest_path}  ({len(manifest_entries)} dataset clips)")

wrote /home/david/Projects/ANIMA_Microtonal_GPT/dataset/listening_test/manifest.json  (14 dataset clips)


In [91]:
# Inline QC — play each clip once to check whether any needs re-selecting.
# for e in manifest_entries:
#     print(f"{e['id']}  {e['song']:24s} [{e['transformation']:17s}] "
#           f"{e['style']:18s} {e['n_chords']:>3d} chords  {e['duration_sec']:5.1f} s")
#     display(Audio(str(ROOT / e["audio"])))

## Next steps

1. **Listen through all 14 clips** above. If any sound muddy / too sparse / repetitive, swap its entry in `SELECTION` — keep the transformation column covered.
2. **Model clips (14)**: once Model B finishes training, `14_listening_test_models.ipynb` will generate 7 clips from Model A and 7 from Model B under matched conditions and append to this manifest.
3. **Randomisation**: the final survey form will shuffle the 28-clip ordering per participant using `manifest.clips` + a participant seed so `source` stays blind.